# ARTI 503 – Stage 1: Sequential NDVI Benchmark

**Everyone runs this notebook on their own laptop / lab PC.**

$$NDVI = \frac{NIR - Red}{NIR + Red}$$

### What you do 
1. Change `MEMBER` in the cell below, then click **Run All**.
2. Copy the line printed in the last cell and paste it into **your row** of the team's Google Sheet. Then fill in your CPU details (instructions at the end).

Before running: **plug in your laptop** and close other heavy apps.
Needs: `pip install numpy matplotlib jupyter`

## Settings

In [ ]:
MEMBER = 1                  # <-- your member number (1-10)

SIZES = [1024, 2048, 4096]  # image sizes to test (pixels per side)
TILE = 512                  # each image is processed in 512 x 512 tiles
RUNS = 5                    # timed runs per size (plus 1 untimed warm-up)

## 1. Load the data
The whole file is loaded into memory **before** timing, so we time only the NDVI calculation.

In [ ]:
from pathlib import Path

# Works whether Jupyter was started in the project folder or in src/
PROJECT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA = PROJECT / "data"

import time
import platform
import os
import numpy as np

data = np.load(DATA / "ndvi_input_4096.npz")
RED, NIR = data["red"], data["nir"]
QUANT = float(data["quant"])
RED_OFFSET, NIR_OFFSET = float(data["red_offset"]), float(data["nir_offset"])
print("Scene:", data["product"])
print("Loaded:", RED.shape, RED.dtype, "| scaling: (DN +", RED_OFFSET, ") /", QUANT)

## 2. The NDVI function
1. Convert the raw numbers (DN) to reflectance: (DN + offset) / 10000.
2. Apply the NDVI formula to every pixel at once (NumPy).
3. Mark bad pixels as empty (NaN): DN = 0 means "no data", and NIR + Red = 0 would divide by zero.

In [ ]:
def compute_ndvi(red_dn, nir_dn):
    red = (red_dn.astype(np.float32) + RED_OFFSET) / QUANT
    nir = (nir_dn.astype(np.float32) + NIR_OFFSET) / QUANT
    with np.errstate(divide="ignore", invalid="ignore"):
        ndvi = (nir - red) / (nir + red)
    ndvi[(red_dn == 0) | (nir_dn == 0) | (nir + red == 0)] = np.nan
    return ndvi

## 3. Tile-based sequential processing
The image is cut into tiles and processed **one tile after another**.
In Stage 2, this loop is what we split across several CPU cores: every tile is independent.

In [ ]:
def ndvi_sequential(size, tile=TILE):
    red, nir = RED[:size, :size], NIR[:size, :size]      # take a size x size area
    result = np.empty((size, size), dtype=np.float32)
    for r in range(0, size, tile):                       # loop over tile rows
        for c in range(0, size, tile):                   # loop over tile columns
            result[r:r+tile, c:c+tile] = compute_ndvi(red[r:r+tile, c:c+tile],
                                                      nir[r:r+tile, c:c+tile])
    return result


# Check: tile-by-tile must give exactly the same answer as the whole image at once
same = np.array_equal(ndvi_sequential(2048), compute_ndvi(RED[:2048, :2048], NIR[:2048, :2048]),
                      equal_nan=True)
print("Correctness check:", "PASS" if same else "FAIL - tell the coders!")

## 4. Benchmark
Each size runs 1 warm-up (not counted), then 5 timed runs. We report the **median** (middle value), which is robust to one unlucky run.

In [ ]:
results = {}
for size in SIZES:
    ndvi_sequential(size)                                # warm-up
    times = []
    for _ in range(RUNS):
        start = time.perf_counter()
        ndvi_sequential(size)
        times.append(time.perf_counter() - start)
    results[size] = times
    print(f"{size} x {size}: median {np.median(times):.4f} s   "
          f"(min {min(times):.4f}, max {max(times):.4f})   "
          f"{size * size / 1e6 / np.median(times):.0f} million pixels/s")

## 5. Which part takes the most time?
This times each step of the NDVI calculation separately, on the largest size.

In [ ]:
red, nir = RED, NIR
steps = {}
t = time.perf_counter(); r = (red.astype(np.float32) + RED_OFFSET) / QUANT
n = (nir.astype(np.float32) + NIR_OFFSET) / QUANT;     steps["1. Convert DN to reflectance"] = time.perf_counter() - t
t = time.perf_counter(); num, den = n - r, n + r;       steps["2. Subtract / add bands"] = time.perf_counter() - t
t = time.perf_counter()
with np.errstate(divide="ignore", invalid="ignore"):
    ndvi = num / den;                                   steps["3. Divide"] = time.perf_counter() - t
t = time.perf_counter(); ndvi[(red == 0) | (nir == 0) | (den == 0)] = np.nan
steps["4. Mask bad pixels"] = time.perf_counter() - t

total = sum(steps.values())
for name, s in steps.items():
    print(f"{name:<32} {s:.4f} s  {100 * s / total:5.1f}%")

### Optional: why NumPy instead of plain Python loops?
The same formula, written pixel by pixel in plain Python, on a small 256 × 256 area.

In [ ]:
def ndvi_python_loop(red_dn, nir_dn):
    out = np.empty(red_dn.shape, dtype=np.float32)
    for i in range(red_dn.shape[0]):
        for j in range(red_dn.shape[1]):
            r = (int(red_dn[i, j]) + RED_OFFSET) / QUANT
            n = (int(nir_dn[i, j]) + NIR_OFFSET) / QUANT
            ok = red_dn[i, j] != 0 and nir_dn[i, j] != 0 and n + r != 0
            out[i, j] = (n - r) / (n + r) if ok else np.nan
    return out

small_r, small_n = RED[:256, :256], NIR[:256, :256]
t = time.perf_counter(); ndvi_python_loop(small_r, small_n); t_loop = time.perf_counter() - t
t = time.perf_counter(); compute_ndvi(small_r, small_n);    t_np = time.perf_counter() - t
print(f"Python loop: {t_loop:.3f} s | NumPy: {t_np:.5f} s | loop is {t_loop / t_np:.0f}x slower")

## 6. NDVI map (sanity check)
Green = vegetation, red/orange = soil, buildings, water.

In [ ]:
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(6, 5))
    plt.imshow(ndvi_sequential(4096)[::4, ::4], cmap="RdYlGn", vmin=-0.2, vmax=0.9)
    plt.colorbar(label="NDVI"); plt.axis("off"); plt.title("NDVI, 4096 x 4096 area"); plt.show()
except ImportError:
    print("matplotlib not installed - skipping the map (not required)")

## 7. Your results – copy into the Google Sheet

In [ ]:
cpu_name = platform.processor() or platform.machine()
line = [MEMBER, cpu_name, os.cpu_count(), platform.system()] + \
       [round(float(np.median(results[s])), 4) for s in SIZES]
print("Member | CPU (auto) | Logical threads | OS | " + " | ".join(f"Time {s} (s)" for s in SIZES))
print("\nCOPY THIS LINE (it pastes into separate cells):\n")
print("\t".join(str(x) for x in line))

### Then fill in your CPU details in the same row
The automatic CPU name is often vague, so look up the exact details:

* **Windows:** Task Manager → *Performance* → *CPU*. The name is at the top right; *Cores*, *Logical processors*, *Base speed* and *L1/L2/L3 cache* are at the bottom.
* **Mac:** Apple menu → *About This Mac* (chip name), then search the chip name online for the cores and cache.
* **Linux:** run `lscpu` in a terminal.

Sheet columns: **Member · CPU model · Cores · Threads · Base clock (GHz) · Max clock (GHz) · L3 cache · RAM (GB) · OS · Time 1024 · Time 2048 · Time 4096 · Notes**